# SentinelAI — 02 Preprocessing (Train / Validation / Test)

هذه النسخة تمنع استخدام مجموعة الاختبار في اختيار المودل.

التقسيم:
- **80% Train**
- **10% Validation**
- **10% Test**

الـScaler وClass Weights يتم تعلمهما من **Train فقط**.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

RANDOM_STATE = 42

cwd = Path.cwd()
BASE_DIR = cwd.parent if cwd.name == "notebooks" else cwd
DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

csv_files = sorted(DATA_DIR.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError("No CSV files found in data/.")

df = pd.concat([pd.read_csv(p, low_memory=False) for p in csv_files], ignore_index=True)
df.columns = df.columns.str.strip()

if "Label" not in df.columns:
    raise KeyError("Label column not found.")

numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
df = df.dropna().drop_duplicates().reset_index(drop=True)

print(f"Clean shape: {df.shape[0]:,} × {df.shape[1]}")

Clean shape: 1,135,409 × 79


In [2]:
# Preserve textual labels when available; otherwise retain the existing numeric coding honestly.
label_numeric = pd.to_numeric(df["Label"], errors="coerce")
if label_numeric.notna().all():
    if not np.allclose(label_numeric, np.round(label_numeric)):
        raise ValueError("Numeric labels are not integer-like.")
    y = label_numeric.astype(int)
    label_mapping = {
        int(c): ("Benign" if int(c) == 1 else f"Attack_{int(c)}")
        for c in sorted(y.unique())
    }
    semantic_mapping_verified = False
else:
    raw_names = df["Label"].astype(str).str.strip()
    unique_names = list(pd.unique(raw_names))
    benign = [n for n in unique_names if n.casefold() == "benign"]
    attacks = [n for n in unique_names if n.casefold() != "benign"]

    name_to_code = {}
    label_mapping = {}
    if benign:
        name_to_code[benign[0]] = 1
        label_mapping[1] = benign[0]
    next_code = 2
    for name in attacks:
        name_to_code[name] = next_code
        label_mapping[next_code] = name
        next_code += 1

    y = raw_names.map(name_to_code).astype(int)
    semantic_mapping_verified = True

print("Labels:")
for code_, name_ in sorted(label_mapping.items()):
    print(f"{code_:>2} -> {name_:<30} {(y == code_).sum():,}")
print("semantic_mapping_verified =", semantic_mapping_verified)

Labels:
 1 -> Benign                         880,060
 2 -> Attack_2                       35,127
 3 -> Attack_3                       33,817
 4 -> Attack_4                       124,280
 5 -> Attack_5                       52,051
 6 -> Attack_6                       7,598
 7 -> Attack_7                       2,028
 8 -> Attack_8                       301
 9 -> Attack_9                       91
10 -> Attack_10                      46
11 -> Attack_11                      10
semantic_mapping_verified = False


In [3]:
# Keep only numeric model features; explicitly exclude row IDs and target columns.
non_features = {"Unnamed: 0", "Label", "Label_Name", "_Traffic"}
candidate_cols = [c for c in df.columns if c not in non_features]
feature_cols = [c for c in candidate_cols if pd.api.types.is_numeric_dtype(df[c])]

excluded_non_numeric = [c for c in candidate_cols if c not in feature_cols]
if excluded_non_numeric:
    print("Excluded non-numeric columns:", excluded_non_numeric)

X = df[feature_cols].copy()
indices = np.arange(len(df))

print("Features:", len(feature_cols))
print("Rows:", len(X))

Features: 77
Rows: 1135409


In [4]:
# First split: 80% train, 20% temporary.
X_train_raw, X_temp_raw, y_train, y_temp, idx_train, idx_temp = train_test_split(
    X, y, indices,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

# Second split: temporary 20% -> 10% validation + 10% test.
X_val_raw, X_test_raw, y_val, y_test, idx_val, idx_test = train_test_split(
    X_temp_raw, y_temp, idx_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_STATE,
)

print("Train:", X_train_raw.shape, y_train.shape)
print("Validation:", X_val_raw.shape, y_val.shape)
print("Test:", X_test_raw.shape, y_test.shape)

assert set(idx_train).isdisjoint(set(idx_val))
assert set(idx_train).isdisjoint(set(idx_test))
assert set(idx_val).isdisjoint(set(idx_test))
print("Split overlap check: PASS")

Train: (908327, 77) (908327,)
Validation: (113541, 77) (113541,)
Test: (113541, 77) (113541,)


Split overlap check: PASS


In [5]:
# Fit preprocessing on TRAIN ONLY.
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
X_val = scaler.transform(X_val_raw).astype(np.float32)
X_test = scaler.transform(X_test_raw).astype(np.float32)

classes = np.sort(y_train.unique())
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weights = {int(c): float(w) for c, w in zip(classes, weights)}

print("Class weights:")
print(class_weights)

Class weights:
{1: 0.11728629556249263, 2: 2.938513996590222, 3: 3.0522355961477716, 4: 0.830535703835913, 5: 1.9830259075954424, 6: 13.58591342846032, 7: 50.90948324178904, 8: 342.63560920407394, 9: 1131.1668742216686, 10: 2231.7616707616708, 11: 10321.897727272728}


In [6]:
# Save reproducible artifacts.
np.save(PROCESSED_DIR / "X_train.npy", X_train)
np.save(PROCESSED_DIR / "X_val.npy", X_val)
np.save(PROCESSED_DIR / "X_test.npy", X_test)
np.save(PROCESSED_DIR / "y_train.npy", y_train.to_numpy(dtype=np.int64))
np.save(PROCESSED_DIR / "y_val.npy", y_val.to_numpy(dtype=np.int64))
np.save(PROCESSED_DIR / "y_test.npy", y_test.to_numpy(dtype=np.int64))

np.save(PROCESSED_DIR / "train_indices.npy", idx_train)
np.save(PROCESSED_DIR / "val_indices.npy", idx_val)
np.save(PROCESSED_DIR / "test_indices.npy", idx_test)

joblib.dump(scaler, PROCESSED_DIR / "scaler.joblib")

(PROCESSED_DIR / "feature_names.json").write_text(
    json.dumps(feature_cols, ensure_ascii=False, indent=2), encoding="utf-8"
)
(PROCESSED_DIR / "label_mapping.json").write_text(
    json.dumps({str(k): v for k, v in label_mapping.items()}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
(PROCESSED_DIR / "class_weights.json").write_text(
    json.dumps({str(k): v for k, v in class_weights.items()}, indent=2),
    encoding="utf-8",
)

manifest = {
    "methodology_version": 2,
    "random_state": RANDOM_STATE,
    "split": {"train": 0.80, "validation": 0.10, "test": 0.10},
    "rows": {
        "train": int(len(y_train)),
        "validation": int(len(y_val)),
        "test": int(len(y_test)),
    },
    "n_features": len(feature_cols),
    "semantic_attack_mapping_verified": semantic_mapping_verified,
    "scaler_fit_on": "train_only",
}
(PROCESSED_DIR / "split_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

print(json.dumps(manifest, ensure_ascii=False, indent=2))

{
  "methodology_version": 2,
  "random_state": 42,
  "split": {
    "train": 0.8,
    "validation": 0.1,
    "test": 0.1
  },
  "rows": {
    "train": 908327,
    "validation": 113541,
    "test": 113541
  },
  "n_features": 77,
  "semantic_attack_mapping_verified": false,
  "scaler_fit_on": "train_only"
}


In [7]:
# Distribution audit — especially important for rare classes.
audit = []
for code_ in sorted(label_mapping):
    audit.append({
        "code": code_,
        "name": label_mapping[code_],
        "train": int((y_train == code_).sum()),
        "validation": int((y_val == code_).sum()),
        "test": int((y_test == code_).sum()),
    })
pd.DataFrame(audit)

,code,name,train,validation,test
0,1,Benign,704048,88006,88006
1,2,Attack_2,28101,3513,3513
2,3,Attack_3,27054,3381,3382
3,4,Attack_4,99424,12428,12428
4,5,Attack_5,41641,5205,5205
5,6,Attack_6,6078,760,760
6,7,Attack_7,1622,203,203
7,8,Attack_8,241,30,30
8,9,Attack_9,73,9,9
9,10,Attack_10,37,5,4


## قاعدة التقييم من الآن

`X_test / y_test` **لن تُستخدم** في ملفات المقارنة (`03`, `05`) ولا في اختيار hyperparameters.
يتم فتحها في `06_tuning.ipynb` فقط بعد تثبيت الاختيار النهائي بناءً على Validation/CV.